# Meta-NATH CAD Full Demo Kaggle Workflow

Thin Kaggle orchestration notebook for branch `taitrn`.

The real logic stays in `.py` and `.sh` files. This notebook clones the repo, links datasets, runs the full 3-tier v1 demo, prints reports, and packages artifacts.

## Before Running

- Kaggle accelerator: GPU.
- Internet: on, unless the repo/model cache is mounted as Kaggle inputs.
- MVTec input should contain category folders like `bottle`, `cable`, `capsule`, ... either directly or one folder down.
- DINOv2 is intentional for this project stage. DINOv3 is explicitly out of scope for this v1 demo.
- The full demo tiers are: main conservative benchmark, mechanism smoke, and experimental NSP2/CBP benchmark.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from tqdm.auto import tqdm

REPO_URL = "https://github.com/taitrn/NestedLearningForCAD.git"
BRANCH = "taitrn"
REPO = Path("/kaggle/working/NestedLearningForCAD")

# Change these if your Kaggle input names differ.
MVTEC_INPUT = Path(os.environ.get("MVTEC_INPUT", "/kaggle/input/mvtec-ad"))

# Full demo defaults. Set MAIN_MAX_TASKS=15 for the heavier full-MVTec pass.
MAIN_MAX_TASKS = int(os.environ.get("MAIN_MAX_TASKS", os.environ.get("MAX_TASKS", "8")))
EXPERIMENTAL_MAX_TASKS = int(os.environ.get("EXPERIMENTAL_MAX_TASKS", str(MAIN_MAX_TASKS)))

RUN_MAIN = os.environ.get("RUN_MAIN", "1")
RUN_MECHANISM = os.environ.get("RUN_MECHANISM", "1")
RUN_EXPERIMENTAL = os.environ.get("RUN_EXPERIMENTAL", "1")
REQUIRE_EXPERIMENTAL_ACCEPTED = os.environ.get("REQUIRE_EXPERIMENTAL_ACCEPTED", "0")

# Kaggle GPU config increases batch/worker/prefetch for the conservative path.
MAIN_CONFIG = os.environ.get("MAIN_CONFIG", "conf/config_phase3_kaggle_gpu.yaml")
CONSERVATIVE_CONFIG = os.environ.get("CONSERVATIVE_CONFIG", MAIN_CONFIG)
EXPERIMENTAL_CONFIG = os.environ.get("EXPERIMENTAL_CONFIG", "conf/config_phase3_experimental_nsp2_cbp.yaml")
FULL_DEMO_SCRIPT = "scripts/run_full_demo.sh"

INSTALL_DEPS = False
INCLUDE_CHECKPOINTS_IN_ZIP = False

def run(cmd, cwd=None, env=None, heartbeat=60):
    cmd = [str(x) for x in cmd]
    start = time.time()
    next_beat = heartbeat
    print(f"\n[{time.strftime('%H:%M:%S')}] $ {' '.join(cmd)}", flush=True)
    proc = subprocess.Popen(cmd, cwd=str(cwd) if cwd else None, env=env)
    while proc.poll() is None:
        elapsed = int(time.time() - start)
        if heartbeat and elapsed >= next_beat:
            print(f"[{time.strftime('%H:%M:%S')}] still running ({elapsed}s): {' '.join(cmd[:3])}", flush=True)
            next_beat += heartbeat
        time.sleep(2)
    elapsed = int(time.time() - start)
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)
    print(f"[{time.strftime('%H:%M:%S')}] done ({elapsed}s)", flush=True)

print("repo:", REPO)
print("branch:", BRANCH)
print("mvtec_input:", MVTEC_INPUT)
print("main_max_tasks:", MAIN_MAX_TASKS)
print("experimental_max_tasks:", EXPERIMENTAL_MAX_TASKS)
print("run_main:", RUN_MAIN)
print("run_mechanism:", RUN_MECHANISM)
print("run_experimental:", RUN_EXPERIMENTAL)
print("main_config:", MAIN_CONFIG)
print("conservative_config:", CONSERVATIVE_CONFIG)
print("experimental_config:", EXPERIMENTAL_CONFIG)



## Clone Or Update Repo

In [ ]:
if REPO.exists():
    run(["git", "fetch", "origin", BRANCH], cwd=REPO)
    run(["git", "checkout", BRANCH], cwd=REPO)
    run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO)
else:
    run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, REPO])

print("Repo ready:", REPO)

## Install Dependencies And Link Dataset

In [ ]:
if INSTALL_DEPS:
    run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=REPO)
else:
    print("Skipping pip install. Set INSTALL_DEPS=True if Kaggle image lacks dependencies.")

import tarfile
import urllib.error
import urllib.request

MVTEC_CATS = ["bottle", "cable", "capsule", "carpet", "grid", "hazelnut", "leather", "metal_nut"]
MVTEC_ARCHIVE_NAMES = ["mvtec_anomaly_detection.tar.xz", "mvtec.tar.xz"]
MVTEC_URLS = [
    "https://huggingface.co/datasets/micguida1/mvtech_anomaly_detection/resolve/main/mvtec_anomaly_detection.tar.xz",
    "https://www.mvtec.com/fileadmin/Redaktion/mvtec.com/company/research/datasets/mvtec_anomaly_detection.tar.xz",
]

def has_categories(path, cats):
    return path.exists() and all((path / cat).is_dir() for cat in cats[:3])

def find_dataset_root(preferred, cats):
    candidates = []
    if preferred.exists():
        candidates.append(preferred)
        candidates.extend([p for p in preferred.iterdir() if p.is_dir()])
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for ds in input_root.iterdir():
            if ds.is_dir():
                candidates.append(ds)
                candidates.extend([p for p in ds.iterdir() if p.is_dir()])
    seen = set()
    for candidate in candidates:
        resolved = candidate.resolve()
        if resolved in seen:
            continue
        seen.add(resolved)
        if has_categories(candidate, cats):
            return candidate
    raise FileNotFoundError(f"Could not find dataset root under /kaggle/input with categories: {cats[:3]}")

def find_mvtec_archive():
    input_root = Path("/kaggle/input")
    if not input_root.exists():
        return None
    for name in MVTEC_ARCHIVE_NAMES:
        matches = list(input_root.rglob(name))
        if matches:
            return matches[0]
    matches = list(input_root.rglob("*mvtec*.tar.xz"))
    return matches[0] if matches else None

def safe_extract_tar(archive, extract_to):
    extract_to.mkdir(parents=True, exist_ok=True)
    root = extract_to.resolve()
    with tarfile.open(archive, "r:xz") as tar:
        members = tar.getmembers()
        total = len(members)
        for member in members:
            member_path = (extract_to / member.name).resolve()
            if root != member_path and root not in member_path.parents:
                raise RuntimeError(f"Unsafe path in tar archive: {member.name}")
        for idx, member in enumerate(members, start=1):
            tar.extract(member, extract_to)
            if idx == 1 or idx == total or idx % 500 == 0:
                print(f"  extracted {idx}/{total}: {member.name[:80]}", flush=True)

def download_mvtec_archive(target):
    target.parent.mkdir(parents=True, exist_ok=True)
    last_error = None
    for url in MVTEC_URLS:
        try:
            print("Downloading MVTec from:", url)
            request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(request, timeout=60) as response, target.open("wb") as out:
                total = int(response.info().get("Content-Length", 0) or 0)
                done = 0
                while True:
                    chunk = response.read(1024 * 1024)
                    if not chunk:
                        break
                    out.write(chunk)
                    done += len(chunk)
                    if total:
                        print(f"  {done / (1024 ** 2):.1f}/{total / (1024 ** 2):.1f} MB", end="\r")
            print("\nDownloaded:", target)
            return target
        except (urllib.error.HTTPError, urllib.error.URLError, TimeoutError, OSError) as exc:
            last_error = exc
            if target.exists():
                target.unlink()
            print("Download failed:", exc)
    raise RuntimeError(f"All MVTec download URLs failed. Last error: {last_error}")

def prepare_mvtec_dataset():
    try:
        return find_dataset_root(MVTEC_INPUT, MVTEC_CATS)
    except FileNotFoundError:
        pass

    extract_to = Path("/kaggle/working/mvtec_extracted")
    if not has_categories(extract_to, MVTEC_CATS):
        archive = find_mvtec_archive()
        if archive is None:
            archive = download_mvtec_archive(Path("/kaggle/working/mvtec_anomaly_detection.tar.xz"))
        else:
            print("Using uploaded MVTec archive:", archive)
        print("Extracting MVTec archive to:", extract_to)
        safe_extract_tar(archive, extract_to)

    return find_dataset_root(extract_to, MVTEC_CATS)

def link_dataset(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        print("Dataset target already exists:", dst)
        return
    try:
        os.symlink(src, dst, target_is_directory=True)
        print("Symlinked", src, "->", dst)
    except OSError:
        print("Symlink failed; copying dataset. This may take a while.")
        shutil.copytree(src, dst)

mvtec_root = prepare_mvtec_dataset()
link_dataset(mvtec_root, REPO / "data" / "mvtec")
print("MVTec root:", mvtec_root)



## Sanity Check

In [ ]:
print("Python:", sys.executable)
run([sys.executable, "-m", "py_compile", "training/run_experiment.py", "training/consolidation_engine.py", "training/meta_nath_engine.py"], cwd=REPO)
run([sys.executable, "-m", "py_compile", "scripts/run_phase3_consolidation.py", "scripts/evaluate_checkpoint.py", "scripts/phase3_acceptance.py", "scripts/compare_checkpoint_scores.py"], cwd=REPO)
run(["bash", "-n", FULL_DEMO_SCRIPT], cwd=REPO)
print("Preflight OK. Mechanism smoke runs inside the full demo when RUN_MECHANISM=1.", flush=True)

## Run Full 3-Tier Demo

This calls `scripts/run_full_demo.sh`. By default it runs the accepted conservative path, mechanism smoke, and experimental NSP2/CBP before/after acceptance benchmark.

In [ ]:
env = os.environ.copy()
env.update({
    "PYTHON_BIN": sys.executable,
    "RUN_MAIN": RUN_MAIN,
    "RUN_MECHANISM": RUN_MECHANISM,
    "RUN_EXPERIMENTAL": RUN_EXPERIMENTAL,
    "MAIN_MAX_TASKS": str(MAIN_MAX_TASKS),
    "EXPERIMENTAL_MAX_TASKS": str(EXPERIMENTAL_MAX_TASKS),
    "MAIN_CONFIG": MAIN_CONFIG,
    "CONSERVATIVE_CONFIG": CONSERVATIVE_CONFIG,
    "EXPERIMENTAL_CONFIG": EXPERIMENTAL_CONFIG,
    "REQUIRE_EXPERIMENTAL_ACCEPTED": REQUIRE_EXPERIMENTAL_ACCEPTED,
    "PROGRESS": "1",
})
run(["bash", FULL_DEMO_SCRIPT], cwd=REPO, env=env)




## Inspect Full Demo Manifest

In [ ]:
from tqdm.auto import tqdm

def as_repo_path(path_text):
    path = Path(path_text)
    return path if path.is_absolute() else REPO / path

def latest_file(pattern):
    print(f"Searching: {pattern}", flush=True)
    files = []
    for path in tqdm(REPO.glob(pattern), desc="matching files", unit="file"):
        files.append(path)
    files = sorted(files, key=lambda p: p.stat().st_mtime)
    if not files:
        raise FileNotFoundError(pattern)
    print(f"Found {len(files)} match(es); using latest: {files[-1]}", flush=True)
    return files[-1]

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

with tqdm(total=5, desc="Inspect full demo artifacts", unit="step") as pbar:
    manifest_path = latest_file("results/MetaNATH_FullDemo_*/manifest.json")
    pbar.update(1)

    manifest = read_json(manifest_path)
    pbar.update(1)

    reports = {}
    for tier_name in ["main_reportable", "experimental_nsp2_cbp"]:
        report_text = manifest["tiers"][tier_name].get("acceptance_report")
        if report_text:
            report_path = as_repo_path(report_text)
            if report_path.exists():
                reports[tier_name] = {"path": str(report_path), "report": read_json(report_path)}
    pbar.update(1)

    summaries = {}
    for tier_name, tier in manifest["tiers"].items():
        phase3_dir = tier.get("phase3_dir")
        if phase3_dir:
            summary_path = as_repo_path(phase3_dir) / "phase3_summary.json"
            if summary_path.exists():
                summaries[tier_name] = {"path": str(summary_path), "summary": read_json(summary_path)}
    pbar.update(1)

    print("manifest:", manifest_path)
    print(json.dumps(manifest, indent=2)[:6000])
    for tier_name, payload in reports.items():
        report = payload["report"]
        print(f"\n{tier_name} acceptance:", payload["path"])
        print("decision:", report.get("decision"), "accepted:", report.get("accepted"))
        print(json.dumps(report.get("checks", []), indent=2))
    for tier_name, payload in summaries.items():
        print(f"\n{tier_name} phase3 summary:", payload["path"])
        print(json.dumps(payload["summary"], indent=2)[:3000])
    pbar.update(1)

## Task Scaling Note

For a stronger MVTec demo, set `MAIN_MAX_TASKS=15` and `EXPERIMENTAL_MAX_TASKS=15` in the first cell or Kaggle environment, then rerun the full demo cell.

In [ ]:
print("Current MAIN_MAX_TASKS:", MAIN_MAX_TASKS)
print("Current EXPERIMENTAL_MAX_TASKS:", EXPERIMENTAL_MAX_TASKS)
print("For 15-task MVTec, set both values to 15 and rerun the full demo cell.")




## Out-Of-Scope Dataset Note

This notebook is the MVTec full-demo orchestrator. VisA remains prepared in `scripts/run_server_visa.sh`, but is not part of this v1 closure run.

In [ ]:
print("VisA is prepared but not included in the 3-tier MVTec v1 demo.")
print("Use scripts/run_server_visa.sh separately when a VisA input is mounted.")


## Package Artifacts

The zip includes JSON/YAML/log/text/markdown artifacts. Checkpoints are excluded by default to keep the output small.

In [ ]:
artifact_zip = Path("/kaggle/working/metanath_kaggle_artifacts.zip")
allowed_suffixes = {".json", ".yaml", ".yml", ".log", ".txt", ".md", ".csv"}
if INCLUDE_CHECKPOINTS_IN_ZIP:
    allowed_suffixes.update({".pt", ".pth", ".ckpt"})

candidate_files = []
for root_name in tqdm(["results", "logs"], desc="scan artifact roots", unit="root"):
    root = REPO / root_name
    if not root.exists():
        continue
    for path in tqdm(root.rglob("*"), desc=f"scan {root_name}", unit="file"):
        if path.is_file() and path.suffix.lower() in allowed_suffixes:
            candidate_files.append(path)

with zipfile.ZipFile(artifact_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in tqdm(candidate_files, desc="zip artifacts", unit="file"):
        zf.write(path, path.relative_to(REPO))

print("artifact_zip:", artifact_zip)
print("files:", len(candidate_files))
print("size_mb:", round(artifact_zip.stat().st_size / (1024 * 1024), 2))